# Démo 4 : construire un réseau avec nn.Module

Objectif : voir pourquoi un seul neurone ne suffit plus quand la relation n'est pas une droite, construire un petit réseau avec une couche cachée et une activation, puis l'écrire comme une classe `nn.Module` et l'inspecter.

**Données** : 60 points générés par une formule, qui ne suivent pas une droite.

**Ce que vous allez voir**

- pourquoi un neurone seul ne sait tracer que des droites ;
- comment une couche cachée et une activation permettent de suivre une courbe ;
- écrire un réseau comme une classe `nn.Module`, l'appeler sur un lot et compter ses paramètres.

**Durée prévue** : 10 min.

*Les sorties enregistrées dans ce notebook viennent d'une exécution dans Google Colab sur GPU A100, avec la graine fixée. Sur un autre GPU ou sur CPU, certaines valeurs peuvent légèrement différer : PyTorch ne garantit pas des résultats identiques d'une plateforme à l'autre.*

## 0. Installation

Cette cellule écrit `requirements-colab.txt` (les versions de la formation, alignées sur le runtime Colab), puis l'installe avec uv. Les paquets déjà présents à la bonne version ne sont pas réinstallés. Hors de Colab, elle n'installe rien : l'environnement uv du projet contient déjà ces versions.

In [ ]:
import importlib.util
from pathlib import Path

Path("requirements-colab.txt").write_text("""torch==2.11.0
torchvision==0.26.0
numpy==2.1.3
matplotlib==3.10.0
pillow==11.3.0
scikit-learn==1.6.1
pandas==2.2.2; python_version < "3.13"
pandas==2.2.3; python_version >= "3.13"
pyarrow>=14
tensorboard==2.21.0
onnx==1.23.0
onnxruntime==1.30.0
onnxscript==0.7.2
torchserve==0.12.0
torch-model-archiver==0.12.0
""")

if importlib.util.find_spec("google.colab") is not None:
    %pip install -q --progress-bar off uv
    !uv pip install --system -q -r requirements-colab.txt
else:
    print("Hors Colab : environnement uv du projet, rien à installer.")

Dans Colab, seuls les paquets absents du runtime ou à une autre version sont téléchargés. Si torch ou torchvision venaient à être réinstallés, redémarrez la session (**Exécution > Redémarrer la session**) puis relancez le notebook.

## 1. Des données qui ne suivent pas une droite

**Cette fois, une relation qui n'est pas une droite.**

Dans la réalité, la plupart des relations ne sont pas linéaires : un prix qui plafonne, une demande
saisonnière, et bien sûr le lien entre des pixels et le contenu d'une image. On veut voir ce qu'il faut pour
les apprendre.

Le code :
- `torch.linspace(-3, 3, 60)` : 60 valeurs régulières entre -3 et 3, mises en colonne avec `unsqueeze(1)` ;
- `x + 2 * torch.sin(1.5 * x)` : une montée qui ondule ;
- `0.2 * torch.randn(60, 1)` : un peu de bruit aléatoire, comme dans de vraies mesures.

In [ ]:
import torch

torch.manual_seed(0)
x = torch.linspace(-3, 3, 60).unsqueeze(1)
y = x + 2 * torch.sin(1.5 * x) + 0.2 * torch.randn(60, 1)
print(x.shape, y.shape)

60 points, une valeur d'entrée et une valeur de sortie chacun : `torch.Size([60, 1])`.

## 2. Regarder les données

**On regarde les données avant de choisir un modèle.**

Le code :
- `plt.scatter(x, y, s=10)` : le nuage de points, avec de petits points (`s=10`).

Prédisez : une droite pourra-t-elle suivre ces points ?

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(x, y, s=10)
plt.show()

La grandeur descend, remonte fortement puis redescend : aucune droite ne peut suivre ces virages.

## 3. Une fonction pour entraîner 

**La boucle de la démo 1, rangée dans une fonction pour la réutiliser sur plusieurs modèles.**

On va entraîner trois modèles ; plutôt que de recopier la boucle trois fois, on l'écrit une fois.

Le code :
- `torch.optim.Adam(..., lr=0.01)` : **Adam**, un optimiseur plus robuste que la descente de gradient simple ;
  il adapte lui-même la taille des pas de chaque paramètre ;
- `nn.functional.mse_loss(model(x), y)` : la même erreur quadratique moyenne que `nn.MSELoss()`, écrite comme
  une fonction ;
- le reste est la boucle de la démo 1 : `zero_grad()`, `backward()`, `step()`, 3000 fois.

In [ ]:
import torch.nn as nn


def entrainer(model, epochs=3000):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    for epoch in range(epochs):
        loss = nn.functional.mse_loss(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"erreur finale : {loss.item():.2f}")

Rien à afficher : la fonction est définie. On y reconnaît les cinq lignes de la boucle de la démo 1.

## 4. Une fonction pour tracer 

**Une fonction qui trace les points et, par-dessus, la courbe d'un modèle.**

Le code :
- `with torch.no_grad():` : on trace des prédictions, sans entraîner ;
- `plt.plot(x, model(x), color="red")` : la courbe du modèle.

In [ ]:
def tracer(model):
    with torch.no_grad():
        plt.scatter(x, y, s=10)
        plt.plot(x, model(x), color="red")
    plt.show()

Rien à afficher : la fonction est définie.

## 5. Un seul neurone : une droite

**Premier essai : le neurone seul de la démo 1.**

Le code :
- `nn.Linear(1, 1)` : une entrée, une sortie, `w * x + b` ;
- `entrainer(neurone)` puis `tracer(neurone)` : on l'entraîne et on regarde sa courbe.

Prédisez : quelle forme aura la courbe rouge ?

In [ ]:
torch.manual_seed(0)
neurone = nn.Linear(1, 1)
entrainer(neurone)
tracer(neurone)

Erreur finale 1.99 : le neurone trace la meilleure droite possible, qui traverse le nuage sans en suivre les
virages.

**À retenir** : un seul neurone ne sait faire que des droites.

## 6. Une couche cachée et une activation

**Deuxième essai : une couche cachée de 32 neurones, une activation, et un neurone de sortie.**

Chaque neurone caché calcule sa propre droite ; l'**activation** ReLU la « plie » en remplaçant les
valeurs négatives par 0. En additionnant 32 droites pliées, le neurone de sortie peut dessiner une courbe.

Le code :
- `nn.Sequential(...)` : enchaîne des couches dans l'ordre ; la sortie de l'une est l'entrée de la suivante ;
- `nn.Linear(1, 32)` : une entrée, 32 neurones cachés ;
- `nn.ReLU()` : l'activation, appliquée à chacune des 32 valeurs ;
- `nn.Linear(32, 1)` : les 32 valeurs combinées en une seule sortie.

In [ ]:
torch.manual_seed(0)
reseau = nn.Sequential(
    nn.Linear(1, 32),
    nn.ReLU(), 
    nn.Linear(32, 1))

entrainer(reseau)
tracer(reseau)

Erreur finale 0.03 : la courbe rouge suit les virages. Chaque neurone caché, grâce à la ReLU, ajoute un
coude à la courbe ; 32 neurones suffisent à suivre cette forme.

**À retenir** : couches cachées et activations donnent au réseau sa capacité à apprendre des relations
complexes.

## 7. Et sans activation ?

**Troisième essai : le même réseau, sans ReLU.**

On veut savoir si c'est le nombre de neurones ou l'activation qui fait la différence.

Le code :
- `nn.Sequential(nn.Linear(1, 32), nn.Linear(32, 1))` : deux couches linéaires enchaînées, sans activation.

Prédisez : droite ou courbe ?

In [ ]:
torch.manual_seed(0)
sans_activation = nn.Sequential(nn.Linear(1, 32), nn.Linear(32, 1))
entrainer(sans_activation)
tracer(sans_activation)

Erreur finale 1.99, exactement celle du neurone seul : sans activation, deux couches linéaires enchaînées
restent une droite.

**Piège fréquent** : oublier l'activation entre deux couches `Linear` ; le réseau perd tout son intérêt.

## 8. Le même réseau, écrit comme une classe

**Pour des réseaux plus riches, on écrit une classe qui hérite de `nn.Module`.**

`nn.Sequential` suffit quand les couches s'enchaînent simplement. Une classe permet tout le reste :
des branches, des calculs intermédiaires, des noms parlants. C'est la forme la plus courante en PyTorch.

Le code :
- `class PetitReseau(nn.Module)` : notre modèle hérite de tout ce que sait faire un module PyTorch ;
- `super().__init__()` : ligne obligatoire, qui prépare le module ;
- `__init__` : on y **déclare** les couches, rangées dans des attributs (`self.cachee`, `self.activation`,
  `self.sortie`) ;
- `forward(self, x)` : on y **décrit le calcul**, de l'entrée à la sortie.

In [ ]:
class PetitReseau(nn.Module):
    def __init__(self):
        super().__init__()
        self.cachee = nn.Linear(1, 32)
        self.activation = nn.ReLU()
        self.sortie = nn.Linear(32, 1)

    def forward(self, x):
        x = self.cachee(x)
        x = self.activation(x)
        x = self.sortie(x)
        return x 

Rien à afficher : la classe est définie. Elle décrit exactement le réseau `nn.Sequential` de l'étape 6, avec des
noms pour chaque couche.

## 9. Créer le modèle et l'appeler sur un batch

**On crée le modèle, on l'affiche, et on l'appelle sur un lot de 5 points.**

Un modèle traite toujours un **lot** (batch) d'exemples à la fois, et rend une prédiction par
exemple.

Le code :
- `PetitReseau()` : crée le modèle, avec des poids tirés au hasard ;
- `print(model)` : la liste des couches, avec leurs noms ;
- `model(x[:5])` : on appelle le modèle comme une fonction, jamais `model.forward(x)` directement ;
- `.shape` : la forme de la sortie.

In [ ]:
torch.manual_seed(0)
model = PetitReseau()
print(model)
print(model(x[:5]).shape)

`print(model)` liste les couches avec les noms choisis dans `__init__`. Un lot de 5 points donne
`torch.Size([5, 1])` : une prédiction par point, calculées en une seule fois.

**À retenir** : la première dimension de la sortie est toujours celle du lot.

## 10. Inspecter les paramètres

**`named_parameters()` liste les tenseurs de poids du modèle, avec leur nom et leur forme.**

C'est ce que l'optimiseur reçoit avec `model.parameters()`. Compter les paramètres donne la taille
du modèle.

Le code :
- `for nom, p in model.named_parameters()` : chaque tenseur de paramètres et son nom ;
- `p.numel()` : le nombre de valeurs d'un tenseur ; la somme donne le total.

Prédisez : combien de paramètres au total ?

In [ ]:
for nom, p in model.named_parameters():
    print(nom, tuple(p.shape))
print("total :", sum(p.numel() for p in model.parameters()))

97 paramètres : 32 poids et 32 biais pour la couche cachée, 32 poids et 1 biais pour la sortie.
`named_parameters()` les retrouve tous : c'est ce que l'optimiseur reçoit.

**Essayez** : passez la couche cachée à 64 neurones. Combien de paramètres ?

## À retenir de cette démo

- Sans activation, empiler des couches linéaires donne encore une droite ; l'activation (ReLU) permet au réseau de se courber.
- `nn.Sequential` enchaîne des couches ; une classe `nn.Module` fait de même, avec `__init__` pour les couches et `forward` pour le calcul.
- Un modèle travaille par lot et expose ses paramètres avec `named_parameters()`.